In [1]:
from pathlib import Path

import open3d as o3d
import numpy as np
import torch
from torch.utils.data import DataLoader

from plant_shape_analysis.models.siren import PointCloudSiren

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# Load and visualize a sample leaf point cloud
sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/1_maize_control_plant1_D00_leaf2_dense.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))
leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
leaf_np = np.asarray(leaf_pcd.points)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
o3d.visualization.draw_geometries([leaf_pcd])

Loaded leaf point cloud with 34789 points.


### Visualize current sampling strategy

In [25]:
# Plot off-surface points sampled around the leaf
#sampling_strategy = "prasad"
sampling_strategy = "mixed"
leaf_dataset = PointCloudSiren(point_cloud, len(point_cloud), sampling_strategy=sampling_strategy)
dataloader = DataLoader(leaf_dataset, shuffle=True, batch_size=1, pin_memory=True, num_workers=0)
for data in dataloader:
    print(data)
    break

Off surface sampling strategy:  mixed
[{'coords': tensor([[[-0.0452, -0.0898, -0.0011],
         [-0.0489, -0.1120, -0.7269],
         [-0.0792, -0.0219, -0.2651],
         ...,
         [-0.7983,  0.5857,  0.4379],
         [-0.3534,  0.9529,  0.8964],
         [ 0.5732, -0.7210, -0.9301]]])}, {'sdf': tensor([[[ 0.],
         [ 0.],
         [ 0.],
         ...,
         [-1.],
         [-1.],
         [-1.]]]), 'normals': tensor([[[-0.8917, -0.4455, -0.0804],
         [-0.9151, -0.3934, -0.0885],
         [-0.9977,  0.0484,  0.0469],
         ...,
         [-1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000]]])}]


In [20]:
points = data[0]['coords'].squeeze()
if sampling_strategy in ["uniform"]:
    surface_points, off_surface_points = points[:len(leaf_np)], points[len(leaf_np):]
elif sampling_strategy in ["mixed", "prasad"]:
    surface_points, off_surface_points = points[:len(leaf_np)], points[len(leaf_np):]
    off_surface_points_local, off_surface_points_global = off_surface_points[:len(leaf_np)], off_surface_points[len(leaf_np):]

In [21]:
surface_points.shape, off_surface_points.shape, off_surface_points_local.shape, off_surface_points_global.shape

(torch.Size([34789, 3]),
 torch.Size([69578, 3]),
 torch.Size([34789, 3]),
 torch.Size([34789, 3]))

In [22]:
# Surface points
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(surface_points.numpy()))])

In [23]:
# All off-surface points
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(off_surface_points.numpy()))])

In [24]:
# Off-surface points local
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(off_surface_points_local.numpy()))])

With sampling from Prasad points were too far from the surface, so they could intersect with other parts of a folded leaf, now IGR sampling is used in "mixed" and it's more efficient and correct

### Closer look to IGR sampling strategy

In [ ]:
local_sigma = 0.02
global_sigma = 1.8
igr_close = surface_points + (torch.randn_like(surface_points) * local_sigma)
igr_far = (torch.rand(1, len(leaf_np) // 8, 3) * (global_sigma * 2)) - global_sigma

In [ ]:
points = np.vstack([surface_points, igr_close, igr_far.squeeze().numpy()])
points = np.vstack([surface_points, igr_close])
points = np.vstack([igr_close])

In [ ]:
points.shape

In [ ]:
o3d.visualization.draw_geometries([o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points))])

Close points are much more fine grained, in current sampling strategy the gaussian points are too far away from the surface